In [59]:
import pandas as pd

players_fresh = pd.read_csv('../data/player_values_fresh.csv')
appearances = pd.read_csv('../data/appearances.csv')


players_fresh_games = players_fresh.merge(appearances, how='left', on ='player_id')
print(players_fresh_games.shape[0])
players_fresh_games = players_fresh_games.dropna()
print(players_fresh_games.shape[0])
print(players_fresh_games[players_fresh_games['goals'] > 0][['name', 'goals']].shape[0])



643797
643407
57892


In [60]:
players_stats = appearances.groupby('player_id').agg(
    total_goals=('goals', 'sum'),
    total_assists=('assists', 'sum'),
    total_minutes=('minutes_played', 'sum'),
    total_appearances=('appearance_id', 'count'),
    total_yellow_cards=('yellow_cards', 'sum'),
    total_red_cards=('red_cards', 'sum')
).reset_index()

players = pd.read_csv('../data/players.csv')
players_scored_400 = players_stats[players_stats['total_goals'] > 400]

print(players.merge(players_scored_400, on = 'player_id', how = 'inner')[['player_id', 'name', 'total_goals']])

players_stats['goals_per_90'] = players_stats['total_goals']/players_stats['total_minutes']*90
players_stats['assists_per_90'] = players_stats['total_assists']/players_stats['total_minutes']*90
print(players_stats[players_stats['player_id'] == 8198])
print(players_stats[players_stats['player_id'] == 28003])
print(players_stats[players_stats['player_id'] == 38253])
print(players_stats[players_stats['player_id'] == 132098])
players_fresh_stats = players_fresh.merge(players_stats, on='player_id', how='left')
print('---------------------', players_fresh_stats)

   player_id                name  total_goals
0       8198   Cristiano Ronaldo          434
1      28003        Lionel Messi          458
2      38253  Robert Lewandowski          528
3     132098          Harry Kane          414
     player_id  total_goals  total_assists  total_minutes  total_appearances  \
730       8198          434            114          41420                485   

     total_yellow_cards  total_red_cards  goals_per_90  assists_per_90  
730                  64                4      0.943023        0.247706  
      player_id  total_goals  total_assists  total_minutes  total_appearances  \
2262      28003          458            224          44940                529   

      total_yellow_cards  total_red_cards  goals_per_90  assists_per_90  
2262                  50                1      0.917223        0.448598  
      player_id  total_goals  total_assists  total_minutes  total_appearances  \
3144      38253          528            122          54203             

In [61]:
print(players_fresh_stats.isna().sum())
print(f"Players with no appearance data: {players_fresh_stats['total_goals'].isna().sum()} / {len(players_fresh_stats)}")
print(players_fresh_stats[players_fresh_stats['total_minutes'] == 0])

player_id               0
current_value           0
current_date            0
peak_value              0
peak_date               0
name                    0
days_since_update       0
is_stale                0
total_goals           390
total_assists         390
total_minutes         390
total_appearances     390
total_yellow_cards    390
total_red_cards       390
goals_per_90          390
assists_per_90        390
dtype: int64
Players with no appearance data: 390 / 6005
Empty DataFrame
Columns: [player_id, current_value, current_date, peak_value, peak_date, name, days_since_update, is_stale, total_goals, total_assists, total_minutes, total_appearances, total_yellow_cards, total_red_cards, goals_per_90, assists_per_90]
Index: []


In [62]:
players_fresh_stats = players_fresh_stats.dropna(subset=['total_goals'])
print(f"Final dataset size: {len(players_fresh_stats)}")

Final dataset size: 5615


In [63]:
print(players_fresh_stats.columns.tolist())

players_fresh_stats = players_fresh_stats.merge(players[['player_id', 'date_of_birth', 'position', 'sub_position', 'foot', 'height_in_cm', 'country_of_citizenship']], on='player_id', how='left')

players_fresh_stats['date_of_birth'] = pd.to_datetime(players_fresh_stats['date_of_birth'])
players_fresh_stats['current_date'] = pd.to_datetime(players_fresh_stats['current_date'])
players_fresh_stats['age'] = (players_fresh_stats['current_date'] - players_fresh_stats['date_of_birth']).dt.days / 365.25
print(players_fresh_stats[['name', 'age', 'position', 'sub_position', 'foot', 'height_in_cm']].head(10))
print(players_fresh_stats.isna().sum())

['player_id', 'current_value', 'current_date', 'peak_value', 'peak_date', 'name', 'days_since_update', 'is_stale', 'total_goals', 'total_assists', 'total_minutes', 'total_appearances', 'total_yellow_cards', 'total_red_cards', 'goals_per_90', 'assists_per_90']
                    name        age    position        sub_position   foot  \
0         Artem Pospelov  27.227926  Goalkeeper          Goalkeeper  right   
1         Dedryck Boyata  34.349076    Defender         Centre-Back  right   
2  Kyriakos Papadopoulos  33.111567    Defender         Centre-Back  right   
3          Alfredo Mejía  35.118412    Midfield  Defensive Midfield  right   
4          Alaixys Romao  41.325120    Midfield  Defensive Midfield  right   
5        Nemanja Nikolic  24.249144    Defender          Right-Back  right   
6       Georgios Marinos  25.040383      Attack        Right Winger   left   
7          Gabriel Pires  31.676934    Midfield  Defensive Midfield   left   
8                   Zeca  36.725530   

In [71]:
players_fresh_stats['foot'] = players_fresh_stats['foot'].fillna('unknown')
players_fresh_stats['height_in_cm'] = players_fresh_stats.groupby('position')['height_in_cm'].transform(lambda x: x.fillna(x.median()))
players_fresh_stats = players_fresh_stats.dropna(subset=['age', 'sub_position'])
print(players_fresh_stats.isna().sum())
print(f"Final shape: {players_fresh_stats.shape}")

player_id                 0
current_value             0
current_date              0
peak_value                0
peak_date                 0
name                      0
days_since_update         0
is_stale                  0
total_goals               0
total_assists             0
total_minutes             0
total_appearances         0
total_yellow_cards        0
total_red_cards           0
goals_per_90              0
assists_per_90            0
date_of_birth             0
position                  0
sub_position              0
foot                      0
height_in_cm              0
country_of_citizenship    0
age                       0
dtype: int64
Final shape: (5613, 23)


In [72]:
players_fresh_stats.to_csv('../data/players_fresh_stats.csv', index=False)